In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

# Load environment variables
load_dotenv()

# Get API key from environment
groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found. Please add it to your .env file.")

# Initialize the LLM
llm = ChatGroq(
    temperature=0, 
    groq_api_key=groq_api_key, 
    model_name="meta-llama/llama-4-scout-17b-16e-instruct"
)

In [ ]:
response = llm.invoke("how are you? ...")
print(response.content)

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://careers.nike.com/software-engineer-ii-itc/job/R-55391")
page_data = loader.load().pop().page_content
print(page_data)

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
        """
        ### SCRAPED TEXT FROM WEBSITE:
        {page_data}
        ### INSTRUCTION:
        The scraped text is from the career's page of a website.
        Your job is to extract the job postings and return them in JSON format containing the 
        following keys: `role`, `experience`, `skills` and `description`.
        Only return the valid JSON.
        ### VALID JSON (NO PREAMBLE):    
        """
)

chain_extract = prompt_extract | llm 
res = chain_extract.invoke(input={'page_data':page_data})
type(res.content)

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

In [ ]:
type(json_res)

In [ ]:
import pandas as pd

df = pd.read_csv("my_portfolio.csv")
df

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Initialize the TF-IDF vectorizer
vectorizer = TfidfVectorizer()

# Fit and transform the techstack data
vectors = vectorizer.fit_transform(df['Techstack'])

def get_similar_links(query_skills, n_results=2):
    # Transform the query
    query_vec = vectorizer.transform([query_skills])
    
    # Calculate similarities
    similarities = cosine_similarity(query_vec, vectors)
    
    # Get top matches
    top_indices = similarities[0].argsort()[-n_results:][::-1]
    
    # Return the links
    return [{"links": df.iloc[idx]['Links']} for idx in top_indices]

In [ ]:
job = json_res
job['skills']

In [ ]:
# Extract job information and parse JSON
chain_extract = prompt_extract | llm 
res = chain_extract.invoke(input={'page_data':page_data})
json_parser = JsonOutputParser()
jobs = json_parser.parse(res.content)

# Convert to list if single job
if not isinstance(jobs, list):
    jobs = [jobs]

# Process each job
for job in jobs:
    # Convert skills to string if it's a list
    skills = job['skills']
    if isinstance(skills, list):
        skills = ', '.join(skills)
    
    # Get similar links
    links = get_similar_links(skills)
    
    # Generate email
    prompt_email = PromptTemplate.from_template(
        """
        ### JOB DESCRIPTION:
        {job_description}
        
        ### INSTRUCTION:
        You are Saikat, a business development executive at XYZ COMPANY. XYZ COMPANY is an AI & Software Consulting company dedicated to facilitating
        the seamless integration of business processes through automated tools. 
        Over our experience, we have empowered numerous enterprises with tailored solutions, fostering scalability, 
        process optimization, cost reduction, and heightened overall efficiency. 
        Your job is to write a cold email to the client regarding the job mentioned above describing the capability of XYZ COMPANY 
        in fulfilling their needs.
        Also add the most relevant ones from the following links to showcase XYZ COMPANY portfolio: {link_list}
        Remember you are Saikat, BDE at XYZ COMPANY. 
        Do not provide a preamble.
        ### EMAIL (NO PREAMBLE):
        """
    )

    chain_email = prompt_email | llm
    res = chain_email.invoke({"job_description": str(job), "link_list": links})
    print(res.content)
    print("\n---\n")  # Separator between multiple emails